# Algorithm 20 constrained initialization from oracle prior

This notebook isolates the constrained-basin claim.

We treat the **CrystalFormer-style prior as oracle** for now:

1. `A -> G` is given,
2. CrystalFormer proposes a template-compatible prior `(T, q, F_seed, L_seed)`,
3. we build `F0 = Phi_T(q0)` in payload frame and map it to model frame,
4. we forward-noise to the current full notebook start `t = 1.0` (internal TDM `tau = 2.0`),
5. we run reverse dynamics with Algorithm 20 PPR from that oracle prior.

## Mode A: frozen-lattice coordinate PPR

This notebook currently runs a **frozen-lattice coordinate-only PPR** ablation.
For every reverse step and every PPR call we keep:

`l_t = l_seed`

and we do **not** run any lattice reverse update. The lattice is treated as
conditioning only.

So the current debugging target is:

`p(f_t, v_t | l_seed, a, T)`

and the effective Algorithm 20 objective is the coordinate-only form:

`min_{xi_r, xi_v, q} rho_W(B[D_f(f_t(xi_r, xi_v), v_t(xi_v), stopgrad(l_t), a, t)], Phi_T(q)) + lambda(t) (||xi_r||^2 + ||xi_v||^2)`

The important part is `stopgrad(l_t)`: lattice is conditioning here, not a
projection variable.

For now we deliberately keep the constrained run starting at the full notebook
diffusion start `t = 1.0`. Because the internal TDM horizon is `T = 2`, this
means the actual internal diffusion time is `tau = 2 * 1.0 = 2.0`. We are
**not** doing the smaller `t_start` sweep in this notebook yet.

For PPR acceptance we use late soft projection with small-improvement gates,
and the proximity weighting follows the PPR-style schedule implemented in
Algorithm 20:

`lambda_eff(t, sigma) = lambda0 * t^2 / (4 sigma_eff^2 + 4)`

which mirrors `src/ppr/ppr-main/experiments/data2d/eval_config.py`.



In [ ]:
import os
import time
import math
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

_cwd = Path.cwd().resolve()
if (_cwd / 'configs').exists() and (_cwd / 'src').exists():
    ROOT = _cwd
elif (_cwd.parent / 'configs').exists() and (_cwd.parent / 'src').exists():
    ROOT = _cwd.parent
else:
    ROOT = _cwd

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
import kldmPlus.algorithm20_kldm_ppr_q_witness as algo20_backend
algo19_backend = importlib.reload(algo19_backend)
algo20_backend = importlib.reload(algo20_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case

Algorithm19State = algo19_backend.Algorithm19State
Algorithm20Config = algo20_backend.Algorithm20Config
q_only_witness_fit = algo20_backend.q_only_witness_fit
ppr_kernel_q_witness = algo20_backend.ppr_kernel_q_witness
witness_torus_sin_loss = algo20_backend.witness_torus_sin_loss
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
c_w_ops = algo19_backend.c_w_ops
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
map_model_to_payload_reference_chart = algo19_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo19_backend.map_payload_reference_chart_to_model_frame
torus_rmse = algo19_backend.torus_rmse

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO20_CONSTRAINED_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO20_CONSTRAINED_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_CONSTRAINED_GRAPH_IDS', '2', '2'))
CONSTRAINED_T_START = float(profile_default('KLDM_ALGO20_CONSTRAINED_T_START', '1.0', '1.0'))
INIT_MODES = tuple(v.strip() for v in str(profile_default('KLDM_ALGO20_CONSTRAINED_INIT_MODES', 'crystalformer_oracle,random_template', 'crystalformer_oracle,random_template')).split(',') if v.strip())
ALGO20_Q_ONLY_STEPS = int(profile_default('KLDM_ALGO20_Q_ONLY_STEPS', '100', '150'))
ALGO20_PROJ_STEPS = int(profile_default('KLDM_ALGO20_PROJ_STEPS', '10', '10'))
ALGO20_LR = float(profile_default('KLDM_ALGO20_LR', '1e-2', '1e-2'))
ALGO20_LAMBDA_FLOOR = float(profile_default('KLDM_ALGO20_LAMBDA_FLOOR', '1e-6', '1e-6'))
ALGO20_GRAD_CLIP = float(profile_default('KLDM_ALGO20_GRAD_CLIP', '10.0', '10.0'))
ALGO20_SOFT_ANCHOR_TOL = float(profile_default('KLDM_ALGO20_SOFT_ANCHOR_TOL', '1e-5', '1e-5'))
ALGO20_DENOISER_VARIANT = str(profile_default('KLDM_ALGO20_DENOISER_VARIANT', 'minus', 'minus'))
ALGO20_COORDINATE_SCORE_MODE = str(profile_default('KLDM_ALGO20_COORDINATE_SCORE_MODE', 'direct', 'direct'))
CONSTRAINED_STEPS = int(profile_default('KLDM_ALGO20_CONSTRAINED_STEPS', '800', '500'))
CONSTRAINED_PROJECTION_INTERVAL = int(profile_default('KLDM_ALGO20_CONSTRAINED_PROJECTION_INTERVAL', '25', '25'))
CONSTRAINED_M = int(profile_default('KLDM_ALGO20_CONSTRAINED_M', '1', '1'))
CONSTRAINED_LAMBDA0 = float(profile_default('KLDM_ALGO20_CONSTRAINED_LAMBDA0', '0.3', '0.3'))
CONSTRAINED_TAU = float(profile_default('KLDM_ALGO20_CONSTRAINED_TAU', '0.25', '0.25'))
CONSTRAINED_ROLLBACK = str(profile_default('KLDM_ALGO20_CONSTRAINED_ROLLBACK', 'true', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
RUN_BASELINE = str(profile_default('KLDM_ALGO20_CONSTRAINED_RUN_BASELINE', 'false', 'false')).strip().lower() in {'1', 'true', 'yes', 'on'}
RUN_PPR = str(profile_default('KLDM_ALGO20_CONSTRAINED_RUN_PPR', 'true', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
LATTICE_DEBUG_SCREEN_INIT_MODES = tuple(v.strip() for v in str(profile_default('KLDM_ALGO20_LATTICE_DEBUG_INIT_MODES', 'crystalformer_oracle', 'crystalformer_oracle')).split(',') if v.strip())
LATTICE_DEBUG_SCREEN_T_X100 = parse_int_list(profile_default('KLDM_ALGO20_LATTICE_DEBUG_T_X100', '50,10', '50,10'))
LATTICE_DEBUG_SCREEN_T_VALUES = tuple(float(v) / 100.0 for v in LATTICE_DEBUG_SCREEN_T_X100)
LATTICE_DEBUG_SCREEN_GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_LATTICE_DEBUG_GRAPH_IDS', ','.join(str(v) for v in GRAPH_IDS), ','.join(str(v) for v in GRAPH_IDS)))

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
if not available_graph_ids:
    raise RuntimeError(f'No requested graph ids are present. requested={GRAPH_IDS} available=1..{full_num_graphs}')
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()
print(f'profile={PROFILE} graphs={GRAPH_IDS} t_start={CONSTRAINED_T_START} (internal tau={2.0 * CONSTRAINED_T_START}) init_modes={INIT_MODES}')
print(f'run_baseline={RUN_BASELINE} run_ppr={RUN_PPR}')







MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[2] t_start=1.0 (internal tau=2.0) init_modes=('crystalformer_oracle', 'random_template')
run_baseline=False run_ppr=True


In [2]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    graph_id_set = set(int(v) for v in GRAPH_IDS)
    for graph_idx0, graph_id in enumerate(graph_ids):
        if int(graph_id) not in graph_id_set:
            continue
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=structure,
        standardization='refined',
        symprec=1e-2,
        angle_tolerance=5.0,
    )
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
    }


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State) -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=ALGO20_DENOISER_VARIANT,
        coordinate_score_mode=ALGO20_COORDINATE_SCORE_MODE,
    )


def witness_stats_from_q(z_payload: torch.Tensor, payload, q: torch.Tensor) -> dict[str, float]:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q.reshape(-1), 1.0) if q.numel() else q.reshape(-1)
    z_sym = chart.expand_q(q_eval, device=z_payload.device, dtype=z_payload.dtype)
    return {
        'witness_sin': float(witness_torus_sin_loss(z_payload, z_sym).detach().item()),
        'witness_rmse_payload': float(torus_rmse(z_payload, z_sym).detach().item()),
    }


def make_q_init(payload, *, mode: str, seed: int, graph_id: int, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    total_dof = int(chart.total_dof)
    if total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'chart_q_ref', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random', 'rand'}:
        set_seed(int(seed) + 30000 * int(graph_id))
        return torch.rand((total_dof,), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q init mode: {mode!r}')


def q_only_fit_from_clean(clean_f: torch.Tensor, payload, *, graph_id: int, q_init_mode: str, seed: int, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    q_seed = q_init if q_init is not None else make_q_init(payload, mode=q_init_mode, seed=seed, graph_id=graph_id, dtype=clean_f.dtype, device=clean_f.device)
    before = witness_stats_from_q(z_payload, payload, q_seed)
    fit = q_only_witness_fit(
        z_payload=z_payload,
        payload=payload,
        q_init=q_seed,
        q_init_mode=q_init_mode,
        steps=int(ALGO20_Q_ONLY_STEPS),
        lr=float(ALGO20_LR),
        grad_clip=float(ALGO20_GRAD_CLIP),
    )
    return {'z_payload': z_payload, 'q_seed': q_seed, 'before': before, 'fit': fit}


def config20(*, M: int, lambda0: float, q_init_mode: str, proj_steps: int) -> Algorithm20Config:
    return Algorithm20Config(
        M=int(M),
        proj_steps=int(proj_steps),
        lr=float(ALGO20_LR),
        lambda_noise=float(lambda0),
        lambda_floor=float(ALGO20_LAMBDA_FLOOR),
        grad_clip=float(ALGO20_GRAD_CLIP),
        anchor_mode='soft',
        denoiser_variant=str(ALGO20_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO20_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO20_SOFT_ANCHOR_TOL),
        q_init_mode=str(q_init_mode),
        q_only_steps=int(ALGO20_Q_ONLY_STEPS),
    )



loaded_graphs= [2] sg= [4]


In [3]:
PHASE_LATE_START_HIGH = 0.6
PHASE_LATE_START_MID = 0.3
PHASE_LATE_START_LOW = 0.05


def constrained_phase_schedule(*, t_value: float, base_projection_interval: int = CONSTRAINED_PROJECTION_INTERVAL) -> dict[str, Any]:
    t = float(t_value)
    if t > PHASE_LATE_START_HIGH:
        return {
            'use_ppr': False,
            'projection_interval': int(base_projection_interval),
            'M': 0,
            'lambda0': 0.0,
            'accept_mode': 'skip',
            'accept_factor': 1.0,
            'schedule_label': f'late_skip_high_t={t:.3f}',
        }
    if PHASE_LATE_START_HIGH >= t > PHASE_LATE_START_MID:
        return {
            'use_ppr': True,
            'projection_interval': 25,
            'M': 1,
            'lambda0': 1.0,
            'accept_mode': 'any_improvement',
            'accept_factor': 1.0,
            'schedule_label': f'late_mid_t={t:.3f}_interval=25_M=1_accept<any',
        }
    if PHASE_LATE_START_MID >= t > PHASE_LATE_START_LOW:
        return {
            'use_ppr': True,
            'projection_interval': 10,
            'M': 1,
            'lambda0': 0.3,
            'accept_mode': 'any_improvement',
            'accept_factor': 1.0,
            'schedule_label': f'late_low_t={t:.3f}_interval=10_M=2_accept<any',
        }
    return {
        'use_ppr': True,
        'projection_interval': 5,
        'M': 2,
        'lambda0': 0.1,
        'accept_mode': 'any_improvement_or_feasible',
        'accept_factor': 1.0,
        'schedule_label': f'late_final_t={t:.3f}_interval=5_M=2_accept<any_or_feasible',
    }


def accept_projection(*, before_witness: float, after_witness: float, feasible: bool, rollback: bool, accept_mode: str, accept_factor: float) -> bool:
    if not rollback:
        return True
    before = float(before_witness)
    after = float(after_witness)
    if accept_mode == 'skip':
        return False
    if accept_mode == 'factor':
        return bool(after < float(accept_factor) * before)
    if accept_mode == 'any_improvement':
        return bool(after < before)
    if accept_mode == 'any_improvement_or_feasible':
        return bool((after < before) or feasible)
    raise ValueError(f'Unsupported accept_mode={accept_mode!r}')


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None) -> torch.Tensor:
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    return make_q_init(payload, mode=mode, seed=seed, graph_id=case.graph_id, dtype=dtype, device=device)


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def prepare_constrained_initial_state(case: GraphCase, *, init_mode: str, t_start: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    t_graph = torch.as_tensor([[float(t_start)]], device=f0_model.device, dtype=f0_model.dtype)
    t_nodes = torch.full((int(f0_model.shape[0]),), float(t_start), device=f0_model.device, dtype=f0_model.dtype)
    mode_norm = str(init_mode).strip().lower()
    set_seed(int(seed) + 71000 * int(case.graph_id) + int(round(1000.0 * float(t_start))) + (0 if mode_norm in {'crystalformer_oracle', 'oracle_structure'} else 1))
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state_dict = {
        'f_t': f_t.detach().clone(),
        'v_t': v_t.detach().clone(),
        'l_t': case.gt_l_feature.detach().clone().reshape(1, -1),
        'l0_frozen': case.gt_l_feature.detach().clone().reshape(1, -1),
        'a_t': case.atomic_numbers.detach().clone(),
        'node_index': batch_view.batch.detach().clone(),
        'edge_node_index': batch_view.edge_node_index.detach().clone(),
        'batch': SimpleNamespace(num_atoms=batch_view.num_atoms.detach().clone()),
        'sampling_tdm': runner.model.tdm,
        'score_network': runner.model.score_network,
        'restore_training': False,
        'sampling_time_grid': torch.linspace(float(t_start), 1e-6, int(CONSTRAINED_STEPS) + 1, device=f0_model.device, dtype=f0_model.dtype),
    }
    return {
        'payload': payload,
        'q0': q0.detach().clone(),
        'f0_model': f0_model.detach().clone(),
        'z0_payload': z0_payload.detach().clone(),
        'state_dict': state_dict,
        'noise': {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()},
    }


def iter_manual_sampling_times(*, state_dict):
    grid = state_dict['sampling_time_grid']
    for i in range(len(grid) - 1):
        t_now = float(grid[i].item())
        t_next = float(grid[i + 1].item())
        dt = float(t_now - t_next)
        yield i + 1, SimpleNamespace(
            now=SimpleNamespace(
                graph=torch.as_tensor([[t_now]], device=grid.device, dtype=grid.dtype),
                nodes=torch.full((int(state_dict['f_t'].shape[0]),), t_now, device=grid.device, dtype=grid.dtype),
            ),
            next=SimpleNamespace(
                graph=torch.as_tensor([[t_next]], device=grid.device, dtype=grid.dtype),
                nodes=torch.full((int(state_dict['f_t'].shape[0]),), t_next, device=grid.device, dtype=grid.dtype),
            ),
            dt=dt,
            t_next_float=t_next,
        )


## Constrained reverse comparison

We run two PPR samplers by default: one from `crystalformer_oracle` and one from
`random_template`, both with the same late-soft scheduler.



In [4]:
def run_constrained_reverse_case(case: GraphCase, *, init_mode: str, t_start: float, use_ppr: bool, proj_steps: int, seed: int = SAMPLE_SEED):
    prepared = prepare_constrained_initial_state(case, init_mode=init_mode, t_start=float(t_start), seed=seed)
    payload = prepared['payload']
    state = prepared['state_dict']
    q_live = prepared['q0'].detach().clone() if prepared['q0'].numel() else prepared['q0']
    projection_trace = []
    started = time.perf_counter()
    total_steps = max(len(state['sampling_time_grid']) - 1, 1)

    for step_idx, times in iter_manual_sampling_times(state_dict=state):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            score_v = state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
                t=times.now.nodes,
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_exp_step(
                f_t=state['f_t'],
                v_t=state['v_t'],
                score_v=score_v,
                index=state['node_index'],
                dt=times.dt,
            )
            state['l_t'] = state['l0_frozen'].detach().clone()

        if not use_ppr:
            continue

        scheduled = constrained_phase_schedule(t_value=float(times.t_next_float), base_projection_interval=int(CONSTRAINED_PROJECTION_INTERVAL))
        if (not bool(scheduled['use_ppr'])) or (step_idx % int(scheduled['projection_interval']) != 0):
            continue

        if step_idx % 25 == 0 or step_idx == total_steps:
            elapsed = float(time.perf_counter() - started)
            eta = (elapsed / max(step_idx, 1)) * max(total_steps - step_idx, 0)
            print(f'    step {step_idx}/{total_steps} t={times.t_next_float:.3f} elapsed={elapsed:.1f}s eta={eta:.1f}s')

        algo_state = make_algo_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        )
        clean_before = clean_prediction(algo_state)
        before_fit = q_only_fit_from_clean(clean_before, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=seed, q_init=q_live)
        kernel = ppr_kernel_q_witness(
            state=algo_state,
            payload=payload,
            model=runner.model,
            config=config20(M=int(scheduled['M']), lambda0=float(scheduled['lambda0']), q_init_mode=init_mode, proj_steps=int(proj_steps)),
            q_init=q_live,
        )
        kernel_tail = kernel.logs[-1] if kernel.logs else {}
        after_anchor = float(kernel_tail.get('c_after_witness_sin', float('inf')))
        feasible = bool(kernel_tail.get('soft_anchor_feasible', False))
        accepted = accept_projection(
            before_witness=float(before_fit['fit']['witness_sin']),
            after_witness=float(after_anchor),
            feasible=feasible,
            rollback=bool(CONSTRAINED_ROLLBACK),
            accept_mode=str(scheduled['accept_mode']),
            accept_factor=float(scheduled['accept_factor']),
        )
        if accepted:
            state['f_t'] = kernel.state.f.detach().clone()
            state['v_t'] = kernel.state.v.detach().clone()
            state['l_t'] = state['l0_frozen'].detach().clone()
            q_live = None if kernel.q_star is None else kernel.q_star.detach().clone()

        clean_after = clean_prediction(make_algo_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        ))
        after_fit = q_only_fit_from_clean(clean_after, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=seed, q_init=q_live)
        elapsed_so_far_s = float(time.perf_counter() - started)
        avg_step_time_s = elapsed_so_far_s / max(step_idx, 1)
        remaining_steps = max(total_steps - step_idx, 0)
        est_remaining_s = float(avg_step_time_s * remaining_steps)
        projection_trace.append({
            'test': 'algo20_constrained_oracle_trace',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'init_mode': init_mode,
            'prior_source': 'crystalformer_oracle' if str(init_mode).strip().lower() in {'crystalformer_oracle', 'oracle_structure'} else 'random_template',
            'method': 'ppr_on_facit_em',
            't_start': float(t_start),
            'proj_steps': int(proj_steps),
            'step': int(step_idx),
            't_next': float(times.t_next_float),
            'M': int(scheduled['M']),
            'lambda0': float(scheduled['lambda0']),
            'schedule_label': str(scheduled['schedule_label']),
            'accept_mode': str(scheduled['accept_mode']),
            'accept_factor': float(scheduled['accept_factor']),
            'soft_anchor_feasible': feasible,
            'accepted': bool(accepted),
            'c_before_witness': float(before_fit['fit']['witness_sin']),
            'c_after_project_anchor': float(after_anchor),
            'c_after_redenoise': float(after_fit['fit']['witness_sin']),
            'ppr_beats_renoise': bool(float(after_fit['fit']['witness_sin']) < float(before_fit['fit']['witness_sin'])),
            'witness_rmse_payload': float(kernel_tail.get('witness_rmse_payload', float('nan'))),
            'elapsed_so_far_s': elapsed_so_far_s,
            'estimated_time_left_s': est_remaining_s,
            'status': 'INFO',
            'PASS': True,
        })

    elapsed_s = float(time.perf_counter() - started)
    final_state = make_algo_state_from_raw(
        f=state['f_t'],
        v=state['v_t'],
        l=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=torch.as_tensor([[1e-6]], device=state['f_t'].device, dtype=state['f_t'].dtype),
        t_nodes=torch.full((int(state['f_t'].shape[0]),), 1e-6, device=state['f_t'].device, dtype=state['f_t'].dtype),
    )
    final_clean = clean_prediction(final_state)
    final_fit = q_only_fit_from_clean(final_clean, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=seed, q_init=q_live)
    endpoint = evaluate_arrays(case, state['f_t'], state['l_t'].reshape(-1), state['a_t'])
    feasible_count = sum(bool(row['soft_anchor_feasible']) for row in projection_trace)
    accepted_count = sum(bool(row['accepted']) for row in projection_trace)
    return {
        'test': 'algo20_constrained_oracle_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'init_mode': init_mode,
        'prior_source': 'crystalformer_oracle' if str(init_mode).strip().lower() in {'crystalformer_oracle', 'oracle_structure'} else 'random_template',
        'method': 'ppr_on_facit_em_reverse_from_template_noise' if use_ppr else 'facit_em_reverse_from_template_noise',
        't_start': float(t_start),
        'proj_steps': int(proj_steps) if use_ppr else np.nan,
        'n_steps': int(CONSTRAINED_STEPS),
        'projection_interval': int(CONSTRAINED_PROJECTION_INTERVAL) if use_ppr else np.nan,
        'runtime_s': elapsed_s,
        'projection_count': len(projection_trace) if use_ppr else np.nan,
        'accepted_fraction': float(accepted_count / max(len(projection_trace), 1)) if use_ppr else np.nan,
        'soft_anchor_feasible_fraction': float(feasible_count / max(len(projection_trace), 1)) if use_ppr else np.nan,
        'final_c_model': float(c_w_ops(state['f_t'], payload).detach().item()),
        'final_witness_sin': float(final_fit['fit']['witness_sin']),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }, projection_trace


compare_rows = []
trace_rows = []
run_modes = ((False, True) if (RUN_BASELINE and RUN_PPR) else ((False,) if RUN_BASELINE else (True,)))
total_jobs = len(GRAPH_CASES) * len(INIT_MODES) * len(run_modes)
print(f'constrained_reverse_compare graphs={[case.graph_id for case in GRAPH_CASES]} init_modes={INIT_MODES} run_modes={run_modes} proj_steps={ALGO20_PROJ_STEPS} t_start={CONSTRAINED_T_START}')
job_idx = 0
for case in GRAPH_CASES:
    for init_mode in INIT_MODES:
        for use_ppr in run_modes:
            job_idx += 1
            proj_steps = int(ALGO20_PROJ_STEPS) if use_ppr else 0
            mode_label = 'ppr' if use_ppr else 'baseline'
            print(f'[{job_idx}/{total_jobs}] graph={case.graph_id} init={init_mode} mode={mode_label} start')
            try:
                row, trace = run_constrained_reverse_case(
                    case,
                    init_mode=init_mode,
                    t_start=float(CONSTRAINED_T_START),
                    use_ppr=use_ppr,
                    proj_steps=int(proj_steps),
                    seed=int(SAMPLE_SEED),
                )
                compare_rows.append(row)
                trace_rows.extend(trace)
                print(f'    done graph={case.graph_id} init={init_mode} mode={mode_label} runtime_s={row.get("runtime_s", float("nan")):.1f}')
            except Exception as exc:
                compare_rows.append(error_row(
                    'algo20_constrained_oracle_compare',
                    case.graph_id,
                    exc,
                    'constrained_oracle_reverse',
                    space_group=case.requested_sg,
                    init_mode=init_mode,
                    t_start=float(CONSTRAINED_T_START),
                    proj_steps=int(proj_steps),
                    method='ppr_on_facit_em' if use_ppr else 'facit_em',
                ))

compare_df = dataframe_result('algo20_constrained_oracle_compare', compare_rows)
compare_df = compare_df.sort_values(['graph', 'init_mode', 't_start', 'method']).reset_index(drop=True)
display(compare_df)

trace_df = dataframe_result('algo20_constrained_oracle_trace', trace_rows)
trace_df = trace_df.sort_values(['graph', 'init_mode', 't_start', 'step']).reset_index(drop=True) if not trace_df.empty else trace_df
display(trace_df)



constrained_reverse_compare graphs=[2] init_modes=('crystalformer_oracle', 'random_template') run_modes=(True,) proj_steps=10 t_start=1.0
[1/2] graph=2 init=crystalformer_oracle mode=ppr start


    step 225/500 t=0.550 elapsed=60.7s eta=74.2s
    step 250/500 t=0.500 elapsed=78.2s eta=78.2s
    step 275/500 t=0.450 elapsed=98.6s eta=80.7s
    step 300/500 t=0.400 elapsed=142.9s eta=95.3s
    step 325/500 t=0.350 elapsed=160.4s eta=86.4s
    step 350/500 t=0.300 elapsed=182.6s eta=78.3s
    step 400/500 t=0.200 elapsed=311.9s eta=78.0s
    step 450/500 t=0.100 elapsed=467.4s eta=51.9s
    step 500/500 t=0.000 elapsed=732.8s eta=0.0s
    done graph=2 init=crystalformer_oracle mode=ppr runtime_s=778.8
[2/2] graph=2 init=random_template mode=ppr start
    step 225/500 t=0.550 elapsed=91.8s eta=112.2s
    step 250/500 t=0.500 elapsed=134.5s eta=134.5s
    step 275/500 t=0.450 elapsed=178.5s eta=146.1s
    step 300/500 t=0.400 elapsed=217.4s eta=145.0s
    step 325/500 t=0.350 elapsed=256.7s eta=138.2s
    step 350/500 t=0.300 elapsed=300.8s eta=128.9s
    step 400/500 t=0.200 elapsed=451.1s eta=112.8s
    step 450/500 t=0.100 elapsed=584.2s eta=64.9s
    step 500/500 t=0.000 elaps

,test,graph,space_group,init_mode,prior_source,method,t_start,proj_steps,n_steps,projection_interval,...,accepted_fraction,soft_anchor_feasible_fraction,final_c_model,final_witness_sin,frac_rmse,rmse,match,valid,PASS,status
0,algo20_constrained_oracle_compare,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em_reverse_from_template_noise,1.0,10,500,25,...,1.000000,0.0,0.018384,0.130878,0.173261,NaN,False,True,True,INFO
1,algo20_constrained_oracle_compare,2,4,random_template,random_template,ppr_on_facit_em_reverse_from_template_noise,1.0,10,500,25,...,0.869565,0.0,0.052037,0.151930,0.195024,NaN,False,True,True,INFO


,test,graph,space_group,init_mode,prior_source,method,t_start,proj_steps,step,t_next,...,accepted,c_before_witness,c_after_project_anchor,c_after_redenoise,ppr_beats_renoise,witness_rmse_payload,elapsed_so_far_s,estimated_time_left_s,status,PASS
0,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,225,5.500004e-01,...,True,0.139485,0.133181,0.126365,True,0.133714,73.042396,89.274039,INFO,True
1,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,250,5.000005e-01,...,True,0.129461,0.116008,0.145171,False,0.118119,90.453987,90.453987,INFO,True
2,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,275,4.500006e-01,...,True,0.142067,0.135090,0.135964,True,0.123171,135.926607,111.212679,INFO,True
3,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,300,4.000006e-01,...,True,0.134528,0.127109,0.132723,True,0.119316,156.600089,104.400060,INFO,True
4,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,325,3.500006e-01,...,True,0.133618,0.125791,0.128551,True,0.118657,176.176962,94.864518,INFO,True
5,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,350,3.000007e-01,...,True,0.129840,0.123675,0.132256,False,0.117503,204.890859,87.810368,INFO,True
6,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,360,2.800007e-01,...,True,0.132462,0.128203,0.131606,True,0.119840,230.493670,89.636427,INFO,True
7,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,370,2.600007e-01,...,True,0.130989,0.125647,0.130043,True,0.118467,259.212110,91.074525,INFO,True
8,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,380,2.400008e-01,...,True,0.130245,0.127378,0.128465,True,0.119393,280.678524,88.635323,INFO,True
9,algo20_constrained_oracle_trace,2,4,crystalformer_oracle,crystalformer_oracle,ppr_on_facit_em,1.0,10,390,2.200008e-01,...,True,0.128477,0.126054,0.127441,True,0.118700,304.143005,85.783924,INFO,True


### Cached result of EM baseline
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>test</th>
      <th>graph</th>
      <th>space_group</th>
      <th>init_mode</th>
      <th>prior_source</th>
      <th>method</th>
      <th>t_start</th>
      <th>n_steps</th>
      <th>projection_interval</th>
      <th>runtime_s</th>
      <th>...</th>
      <th>accepted_fraction</th>
      <th>soft_anchor_feasible_fraction</th>
      <th>final_c_model</th>
      <th>final_witness_sin</th>
      <th>frac_rmse</th>
      <th>rmse</th>
      <th>match</th>
      <th>valid</th>
      <th>PASS</th>
      <th>status</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>algo20_constrained_oracle_compare</td>
      <td>2</td>
      <td>4</td>
      <td>crystalformer_oracle</td>
      <td>crystalformer_oracle</td>
      <td>facit_em_reverse_from_template_noise</td>
      <td>1.0</td>
      <td>500</td>
      <td>NaN</td>
      <td>52.299134</td>
      <td>...</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>0.020723</td>
      <td>0.148701</td>
      <td>0.169899</td>
      <td>NaN</td>
      <td>False</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
    </tr>
    <tr>
      <th>1</th>
      <td>algo20_constrained_oracle_compare</td>
      <td>2</td>
      <td>4</td>
      <td>crystalformer_oracle</td>
      <td>crystalformer_oracle</td>
      <td>ppr_on_facit_em_reverse_from_template_noise</td>
      <td>1.0</td>
      <td>500</td>
      <td>25.0</td>
      <td>1591.194690</td>
      <td>...</td>
      <td>1.0</td>
      <td>0.0</td>
      <td>0.015879</td>
      <td>0.104575</td>
      <td>0.176594</td>
      <td>NaN</td>
      <td>False</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
    </tr>
    <tr>
      <th>2</th>
      <td>algo20_constrained_oracle_compare</td>
      <td>2</td>
      <td>4</td>
      <td>random_template</td>
      <td>random_template</td>
      <td>facit_em_reverse_from_template_noise</td>
      <td>1.0</td>
      <td>500</td>
      <td>NaN</td>
      <td>62.674638</td>
      <td>...</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>0.047966</td>
      <td>0.215315</td>
      <td>0.189339</td>
      <td>0.438566</td>
      <td>True</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
    </tr>
    <tr>
      <th>3</th>
      <td>algo20_constrained_oracle_compare</td>
      <td>2</td>
      <td>4</td>
      <td>random_template</td>
      <td>random_template</td>
      <td>ppr_on_facit_em_reverse_from_template_noise</td>
      <td>1.0</td>
      <td>500</td>
      <td>25.0</td>
      <td>1387.568460</td>
      <td>...</td>
      <td>1.0</td>
      <td>0.0</td>
      <td>0.046719</td>
      <td>0.150903</td>
      <td>0.164421</td>
      <td>NaN</td>
      <td>False</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
    </tr>
  </tbody>
</table>
<p>4 rows × 21 columns</p>
</div>

## Reading guide

The key question is whether repeated late coordinate-only PPR updates keep
reducing the witness residual once we freeze the lattice and stay in the seeded
symmetry basin.

The most informative columns are:

- `proj_steps`
- `elapsed_so_far_s`
- `estimated_time_left_s`
- `accepted_fraction`
- `c_before_witness`
- `c_after_project_anchor`
- `c_after_redenoise`
- `ppr_beats_renoise`
- `final_witness_sin`
- `frac_rmse`

Interpretation:

- If `c_after_project_anchor < c_before_witness`, the witness optimizer is doing useful local work.
- If `c_after_redenoise < c_before_witness`, the improvement survives native KLDM renoising.
- If smaller `proj_steps` keep most of the witness gain, then the heavier optimization budget is not worth the runtime.
- If witness improves but `frac_rmse` does not, then the constraint objective is helping symmetry more than endpoint geometry.



## Cheap lattice-mode lightweight probe

Instead of full sampling, we probe a few representative reverse-time states and
run exactly one Algorithm 20 PPR correction in each lattice mode.

Modes:

- **Mode A: frozen lattice**
  - keep `l_t = l_seed`
  - no lattice reverse update
- **Mode B: native lattice**
  - start from `l_seed`
  - let KLDM update `l_t` normally during reverse
  - PPR treats `l_t` as stopgrad conditioning
- **Mode C: consistent noised lattice**
  - forward-noise `l_seed` to `t_start`
  - reverse `l_t` normally
  - PPR still does not project `l_t`

This is a cheap local comparison. The goal is to see which lattice path gives
the best single-step PPR result and reconstructed endpoint metrics.



In [5]:
LATTICE_DEBUG_MODES = ('A_frozen_lattice', 'B_native_lattice', 'C_consistent_noised_lattice')
LATTICE_DEBUG_PROJ_STEPS = int(min(PROJ_STEPS_VALUES)) if len(PROJ_STEPS_VALUES) else 10
LATTICE_DEBUG_CASES = [case for case in GRAPH_CASES if int(case.graph_id) in set(int(v) for v in LATTICE_DEBUG_SCREEN_GRAPH_IDS)]


def prepare_lattice_mode_state(case: GraphCase, *, init_mode: str, lattice_mode: str, t_start: float, seed: int = SAMPLE_SEED):
    prepared = prepare_constrained_initial_state(case, init_mode=init_mode, t_start=float(t_start), seed=seed)
    state = prepared['state_dict']
    mode = str(lattice_mode)
    if mode == 'A_frozen_lattice':
        state['l_t'] = state['l0_frozen'].detach().clone()
    elif mode == 'B_native_lattice':
        state['l_t'] = state['l0_frozen'].detach().clone()
    elif mode == 'C_consistent_noised_lattice':
        t_lattice = torch.as_tensor([float(t_start)], device=state['l_t'].device, dtype=state['l_t'].dtype)
        l_noisy, _eps_l = runner.model.diffusion_l.forward_sample(
            t=t_lattice,
            x0=state['l0_frozen'],
            num_atoms=state['batch'].num_atoms,
        )
        state['l_t'] = l_noisy.detach().clone()
    else:
        raise ValueError(f'Unknown lattice_mode={lattice_mode!r}')
    return prepared


def lattice_mode_reverse_step(*, state, times, lattice_mode: str):
    with torch.no_grad():
        preds_curr = runner.model.score_network(
            t=times.now.graph,
            pos=state['f_t'],
            v=state['v_t'],
            h=state['a_t'],
            l=state['l_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
        )
        score_v = state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=state['v_t'],
            pred_v=preds_curr['v'],
            index=state['node_index'],
        )
        state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_exp_step(
            f_t=state['f_t'],
            v_t=state['v_t'],
            score_v=score_v,
            index=state['node_index'],
            dt=times.dt,
        )
        if lattice_mode == 'A_frozen_lattice':
            state['l_t'] = state['l0_frozen'].detach().clone()
        else:
            state['l_t'] = runner.model.diffusion_l.reverse_step(
                t=times.now.graph.reshape(-1),
                x_t=state['l_t'],
                pred=preds_curr['l'],
                dt=times.dt,
            )


def advance_to_probe(case: GraphCase, *, init_mode: str, lattice_mode: str, probe_t: float, seed: int = SAMPLE_SEED):
    prepared = prepare_lattice_mode_state(case, init_mode=init_mode, lattice_mode=lattice_mode, t_start=float(CONSTRAINED_T_START), seed=seed)
    state = prepared['state_dict']
    selected_times = None
    for step_idx, times in iter_manual_sampling_times(state_dict=state):
        lattice_mode_reverse_step(state=state, times=times, lattice_mode=lattice_mode)
        if float(times.t_next_float) <= float(probe_t):
            selected_times = times
            break
    if selected_times is None:
        selected_times = list(iter_manual_sampling_times(state_dict=state))[-1][1]
    return prepared, state, selected_times


def lightweight_lattice_probe(case: GraphCase, *, init_mode: str, lattice_mode: str, probe_t: float, proj_steps: int, seed: int = SAMPLE_SEED):
    prepared, state, times = advance_to_probe(case, init_mode=init_mode, lattice_mode=lattice_mode, probe_t=float(probe_t), seed=seed)
    payload = prepared['payload']
    q_live = prepared['q0'].detach().clone() if prepared['q0'].numel() else prepared['q0']
    algo_state = make_algo_state_from_raw(
        f=state['f_t'],
        v=state['v_t'],
        l=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=times.next.graph,
        t_nodes=times.next.nodes,
    )
    clean_before = clean_prediction(algo_state)
    before_fit = q_only_fit_from_clean(clean_before, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=seed, q_init=q_live)
    before_endpoint = evaluate_arrays(case, algo_state.f, algo_state.l.reshape(-1), algo_state.atom_types)
    scheduled = constrained_phase_schedule(t_value=float(times.t_next_float), base_projection_interval=int(CONSTRAINED_PROJECTION_INTERVAL))
    kernel = ppr_kernel_q_witness(
        state=algo_state,
        payload=payload,
        model=runner.model,
        config=config20(M=max(int(scheduled['M']), 1), lambda0=float(scheduled['lambda0']), q_init_mode=init_mode, proj_steps=int(proj_steps)),
        q_init=q_live,
    )
    kernel_tail = kernel.logs[-1] if kernel.logs else {}
    after_anchor = float(kernel_tail.get('c_after_witness_sin', float('inf')))
    after_state = make_algo_state_from_raw(
        f=kernel.state.f,
        v=kernel.state.v,
        l=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=times.next.graph,
        t_nodes=times.next.nodes,
    )
    clean_after = clean_prediction(after_state)
    after_fit = q_only_fit_from_clean(clean_after, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=seed, q_init=(kernel.q_star if kernel.q_star is not None else q_live))
    after_endpoint = evaluate_arrays(case, after_state.f, after_state.l.reshape(-1), after_state.atom_types)
    return {
        'test': 'algo20_lattice_mode_probe',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'init_mode': init_mode,
        'lattice_mode': lattice_mode,
        'probe_t': float(times.t_next_float),
        'proj_steps': int(proj_steps),
        'M': max(int(scheduled['M']), 1),
        'lambda0': float(scheduled['lambda0']),
        'c_before_witness': float(before_fit['fit']['witness_sin']),
        'c_after_project_anchor': float(after_anchor),
        'c_after_redenoise': float(after_fit['fit']['witness_sin']),
        'ppr_beats_renoise': bool(float(after_fit['fit']['witness_sin']) < float(before_fit['fit']['witness_sin'])),
        'soft_anchor_feasible': bool(kernel_tail.get('soft_anchor_feasible', False)),
        'witness_rmse_payload': float(kernel_tail.get('witness_rmse_payload', float('nan'))),
        'frac_rmse_before': float(before_endpoint['frac_rmse']),
        'frac_rmse_after': float(after_endpoint['frac_rmse']),
        'rmse_before': float(before_endpoint['rmse']) if before_endpoint['rmse'] is not None else float('nan'),
        'rmse_after': float(after_endpoint['rmse']) if after_endpoint['rmse'] is not None else float('nan'),
        'match_before': bool(before_endpoint['match']),
        'match_after': bool(after_endpoint['match']),
        'valid_before': bool(before_endpoint['valid']),
        'valid_after': bool(after_endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }


probe_rows = []
total_jobs = max(len(LATTICE_DEBUG_CASES), 1) * max(len(LATTICE_DEBUG_SCREEN_INIT_MODES), 1) * len(LATTICE_DEBUG_MODES) * max(len(LATTICE_DEBUG_SCREEN_T_VALUES), 1)
job_idx = 0
print(f'lattice_probe_cases={[case.graph_id for case in LATTICE_DEBUG_CASES]} init_modes={LATTICE_DEBUG_SCREEN_INIT_MODES} probe_t={LATTICE_DEBUG_SCREEN_T_VALUES} proj_steps={LATTICE_DEBUG_PROJ_STEPS}')
for case in LATTICE_DEBUG_CASES:
    for init_mode in LATTICE_DEBUG_SCREEN_INIT_MODES:
        for lattice_mode in LATTICE_DEBUG_MODES:
            for probe_t in LATTICE_DEBUG_SCREEN_T_VALUES:
                job_idx += 1
                print(f'[{job_idx}/{total_jobs}] graph={case.graph_id} init={init_mode} lattice_mode={lattice_mode} probe_t={probe_t:.3f}')
                try:
                    probe_rows.append(lightweight_lattice_probe(
                        case,
                        init_mode=init_mode,
                        lattice_mode=lattice_mode,
                        probe_t=float(probe_t),
                        proj_steps=int(LATTICE_DEBUG_PROJ_STEPS),
                        seed=int(SAMPLE_SEED),
                    ))
                except Exception as exc:
                    probe_rows.append(error_row(
                        'algo20_lattice_mode_probe',
                        case.graph_id,
                        exc,
                        'lattice_mode_lightweight_probe',
                        space_group=case.requested_sg,
                        init_mode=init_mode,
                        lattice_mode=lattice_mode,
                        probe_t=float(probe_t),
                    ))

probe_df = dataframe_result('algo20_lattice_mode_probe', probe_rows)
probe_df = probe_df.sort_values(['graph', 'init_mode', 'lattice_mode', 'probe_t']).reset_index(drop=True)
display(probe_df)



NameError: name 'PROJ_STEPS_VALUES' is not defined

## Lattice conditioning sensitivity of `D_f`

This screening keeps the same sampled `f_t, v_t` state and only changes the
lattice passed into the denoiser.

For a shared sampled state we evaluate `D_f` with:

- `l_GT`
- `l_frozen_seed`
- `l_native`
- `l_consistent_noised`
- `l_random_wrong`

and compare:

- `witness_sin`
- `frac_rmse`
- `validity`

This is a very cheap way to see how much lattice conditioning alone changes the
clean prediction, without rerunning full PPR for every lattice choice.



In [ ]:
LATTICE_COND_INIT_MODES = tuple(v.strip() for v in str(profile_default('KLDM_ALGO20_LATTICE_COND_INIT_MODES', 'crystalformer_oracle', 'crystalformer_oracle')).split(',') if v.strip())
LATTICE_COND_SOURCE_MODES = ('A_frozen_lattice', 'B_native_lattice', 'C_consistent_noised_lattice')
LATTICE_COND_T_VALUES = tuple(float(v) / 100.0 for v in parse_int_list(profile_default('KLDM_ALGO20_LATTICE_COND_T_X100', '20,5', '20,5')))
LATTICE_COND_GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_LATTICE_COND_GRAPH_IDS', ','.join(str(v) for v in GRAPH_IDS), ','.join(str(v) for v in GRAPH_IDS)))
LATTICE_COND_CASES = [case for case in GRAPH_CASES if int(case.graph_id) in set(int(v) for v in LATTICE_COND_GRAPH_IDS)]


def make_random_wrong_lattice(case: GraphCase, *, ref_l: torch.Tensor, seed: int = SAMPLE_SEED) -> torch.Tensor:
    set_seed(int(seed) + 910000 + int(case.graph_id))
    return runner.model.diffusion_l.sample_prior(
        x_like=ref_l.reshape(1, -1),
        num_atoms=torch.tensor([int(case.gt_frac.shape[0])], device=ref_l.device, dtype=torch.long),
    ).detach().clone()


def lattice_eval_variants(case: GraphCase, *, frozen_seed_l: torch.Tensor, native_l: torch.Tensor, consistent_noised_l: torch.Tensor, seed: int = SAMPLE_SEED):
    return {
        'l_GT': case.gt_l_feature.detach().clone().reshape(1, -1).to(device=frozen_seed_l.device, dtype=frozen_seed_l.dtype),
        'l_frozen_seed': frozen_seed_l.detach().clone(),
        'l_native': native_l.detach().clone(),
        'l_consistent_noised': consistent_noised_l.detach().clone(),
        'l_random_wrong': make_random_wrong_lattice(case, ref_l=frozen_seed_l, seed=seed),
    }


def get_probe_state(case: GraphCase, *, init_mode: str, lattice_mode: str, probe_t: float, seed: int = SAMPLE_SEED):
    prepared, state, times = advance_to_probe(case, init_mode=init_mode, lattice_mode=lattice_mode, probe_t=float(probe_t), seed=seed)
    return prepared, state, times


sensitivity_rows = []
total_jobs = max(len(LATTICE_COND_CASES), 1) * max(len(LATTICE_COND_INIT_MODES), 1) * len(LATTICE_COND_SOURCE_MODES) * max(len(LATTICE_COND_T_VALUES), 1)
job_idx = 0
print(f'lattice_conditioning_cases={[case.graph_id for case in LATTICE_COND_CASES]} init_modes={LATTICE_COND_INIT_MODES} probe_t={LATTICE_COND_T_VALUES}')
for case in LATTICE_COND_CASES:
    for init_mode in LATTICE_COND_INIT_MODES:
        for probe_t in LATTICE_COND_T_VALUES:
            cached_states = {}
            for source_mode in LATTICE_COND_SOURCE_MODES:
                job_idx += 1
                print(f'[{job_idx}/{total_jobs}] graph={case.graph_id} init={init_mode} source_mode={source_mode} probe_t={probe_t:.3f}')
                try:
                    prepared, state, times = get_probe_state(case, init_mode=init_mode, lattice_mode=source_mode, probe_t=float(probe_t), seed=int(SAMPLE_SEED))
                    cached_states[source_mode] = (prepared, state, times)
                except Exception as exc:
                    sensitivity_rows.append(error_row(
                        'algo20_lattice_conditioning_probe',
                        case.graph_id,
                        exc,
                        'lattice_conditioning_source_state',
                        space_group=case.requested_sg,
                        init_mode=init_mode,
                        source_lattice_mode=source_mode,
                        probe_t=float(probe_t),
                    ))
            if set(cached_states) != set(LATTICE_COND_SOURCE_MODES):
                continue
            frozen_seed_l = cached_states['A_frozen_lattice'][1]['l0_frozen'].detach().clone()
            native_l = cached_states['B_native_lattice'][1]['l_t'].detach().clone()
            consistent_noised_l = cached_states['C_consistent_noised_lattice'][1]['l_t'].detach().clone()
            eval_ls = lattice_eval_variants(case, frozen_seed_l=frozen_seed_l, native_l=native_l, consistent_noised_l=consistent_noised_l, seed=int(SAMPLE_SEED))
            for source_mode, (prepared, state, times) in cached_states.items():
                payload = prepared['payload']
                q_live = prepared['q0'].detach().clone() if prepared['q0'].numel() else prepared['q0']
                for eval_name, eval_l in eval_ls.items():
                    try:
                        algo_state = make_algo_state_from_raw(
                            f=state['f_t'],
                            v=state['v_t'],
                            l=eval_l,
                            atom_types=state['a_t'],
                            node_index=state['node_index'],
                            edge_node_index=state['edge_node_index'],
                            t_graph=times.next.graph,
                            t_nodes=times.next.nodes,
                        )
                        clean_eval = clean_prediction(algo_state)
                        fit_eval = q_only_fit_from_clean(clean_eval, payload, graph_id=case.graph_id, q_init_mode=init_mode, seed=int(SAMPLE_SEED), q_init=q_live)
                        endpoint_eval = evaluate_arrays(case, clean_eval, eval_l.reshape(-1), state['a_t'])
                        sensitivity_rows.append({
                            'test': 'algo20_lattice_conditioning_probe',
                            'graph': case.graph_id,
                            'space_group': case.requested_sg,
                            'init_mode': init_mode,
                            'source_lattice_mode': source_mode,
                            'probe_t': float(times.t_next_float),
                            'eval_lattice_mode': eval_name,
                            'witness_sin': float(fit_eval['fit']['witness_sin']),
                            'witness_rmse_payload': float(fit_eval['fit']['witness_rmse_payload']),
                            'frac_rmse': float(endpoint_eval['frac_rmse']),
                            'rmse': float(endpoint_eval['rmse']) if endpoint_eval['rmse'] is not None else float('nan'),
                            'match': bool(endpoint_eval['match']),
                            'valid': bool(endpoint_eval['valid']),
                            'PASS': True,
                            'status': 'INFO',
                        })
                    except Exception as exc:
                        sensitivity_rows.append(error_row(
                            'algo20_lattice_conditioning_probe',
                            case.graph_id,
                            exc,
                            'lattice_conditioning_eval',
                            space_group=case.requested_sg,
                            init_mode=init_mode,
                            source_lattice_mode=source_mode,
                            probe_t=float(times.t_next_float),
                            eval_lattice_mode=eval_name,
                        ))

sensitivity_df = dataframe_result('algo20_lattice_conditioning_probe', sensitivity_rows)
sensitivity_df = sensitivity_df.sort_values(['graph', 'init_mode', 'source_lattice_mode', 'probe_t', 'eval_lattice_mode']).reset_index(drop=True)
display(sensitivity_df)

